<a href="https://colab.research.google.com/github/nitin-rajesh/NLP-RL-Pipeline/blob/main/PreprocessDataset_NaturalInstructions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import random
from ast import literal_eval

from datasets import Dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from huggingface_hub import login
from google.colab import userdata
# userdata.get('secretName')
# load from Colab Secrets
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

# ensure all HF libs use this token
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
os.environ["HF_TOKEN"] = hf_token

print("HF auth complete.")


HF auth complete.


In [ ]:
# save as generate_prompts_pipeline.py
import json
import os
import time
from datasets import load_dataset
from tqdm.auto import tqdm
import hashlib

# --- CONFIG ---
DATASET_NAME = "Muennighoff/natural-instructions"
TARGET_COUNT = 1_000_000
SEED = 42
OUTPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgen.jsonl"
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"  # Changed to an accessible model
MAX_NEW_TOKENS = 32000
TEMPERATURE = 0.0
TOP_P = 1.0
BATCH_SIZE = 1
# SAVE_EVERY = 100  # flush to disk every N examples (we append per example anyway)
# ----------------

def extract_json_from_text(text):
    text = text.strip()
    # Try direct JSON first
    try:
        return json.loads(text)
    except Exception:
        pass
    # Otherwise find first '{' and last '}' and try
    start = text.find('{')
    end = text.rfind('}')
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        try:
            return json.loads(candidate)
        except Exception:
            pass
    # fallback: return None
    return None

# Create a deterministic unique key for deduping
def dedupe_key(row):
    # prefer id if present
    if "id" in row and row["id"] not in (None, ""):
        return f"id::{row['id']}"
    # fall back to input (some rows use "input" or "inputs")
    input_field = row.get("input") or row.get("inputs") or ""
    if input_field is None:
        input_field = ""
    # hash to reduce memory footprint
    h = hashlib.sha256(input_field.encode("utf-8")).hexdigest()
    return f"input_hash::{h}"

In [ ]:
# Build the instruction given a dataset row
def build_generation_system_and_user(definition, input_text):


    # )
    system = (
    "You are a professional prompt engineer. For the given task definition and input, "
    "use your world knowledge and editorial judgment to rewrite the task definition into a clearer, "
    "more explicit, more actionable instruction that will yield a high-quality answer from a downstream LLM. "
    "The rewritten instruction must fully replace the original task definition and must be embedded directly inside each prompt.\n\n"

    "Create exactly 3 distinct candidate prompts that will each be fed directly to another LLM. "
    "Each prompt must be self-contained and include:\n"
    "  - the rewritten version of the task definition (as a single integrated instruction),\n"
    "  - guidance or constraints that would help the downstream LLM produce a strong answer,\n"
    "  - and the placeholder or directive for how the downstream model should use the provided input.\n\n"

    "Output ONLY valid JSON with the schema:\n"
    '{ \"prompts\": [ '
    '{\"id\":\"p1\",\"prompt\":\"...\"}, '
    '{\"id\":\"p2\",\"prompt\":\"...\"}, '
    '{\"id\":\"p3\",\"prompt\":\"...\"} '
    '] }\n\n'

    "STRICT RULES:\n"
    "1. Do NOT output anything outside the JSON object.\n"
    "2. The original task definition must NOT appear in the prompts. Only your improved rewritten version should appear Each prompt must reflect your improved version of the task definition—add clarity, remove ambiguity, "
    "and include any constraints, assumptions, or world-knowledge context that improve answer quality.\n"
    "3. Each 'prompt' must be ready to feed directly to a downstream LLM as-is.\n"
    "4. Make the three prompts meaningfully different:\n"
    "   - p1: concise, sharpened, high-precision instruction.\n"
    "   - p2: structured or step-by-step instruction that guides reasoning.\n"
    "   - p3: enriched instruction that incorporates relevant world knowledge, assumptions, or clarifications.\n"
    "5. Include explicit output format requirements inside the prompts when helpful (e.g., JSON schema, bullet list, single sentence, etc.).\n"
    "6. Remeber your task is to improve on the task definition.\n\n"

    "Only output the JSON object."
  )


    user = f"Task definition:\n{definition}\n\nInput:\n{input_text}\n\nProduce JSON now."
    return system, user


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# global single-load cache
_LOCAL_GEN = {"ready": False}

def generate_with_local_model(prompt_text,
                              model_name="mistralai/Mistral-7B-Instruct-v0.3",
                              max_new_tokens=256,
                              temperature=0.0,
                              top_p=1.0):
    """
    MINIMAL version.
    - Loads model ONCE, in FP16 full weights.
    - No try/except anywhere.
    - If anything fails (download, HF token, OOM, missing lib)
      → Python will show the real error.
    """

    global _LOCAL_GEN

    if not _LOCAL_GEN["ready"]:
        # ---------- Load tokenizer ----------
        tokenizer = AutoTokenizer.from_pretrained(
            model_name
        )
        # ensure tokenizer has a pad token
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id


        # ---------- Load model in FP16 (no fallback) ----------
        print(f"[model] Loading {model_name} in FP16...")
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",             # puts layers on your GPU
            trust_remote_code=True,
            use_safetensors=True
        )
        print("[model] Loaded successfully.")

        model.eval()

        _LOCAL_GEN["tokenizer"] = tokenizer
        _LOCAL_GEN["model"] = model
        _LOCAL_GEN["ready"] = True

    # ---------- Generate ----------
    tokenizer = _LOCAL_GEN["tokenizer"]
    model = _LOCAL_GEN["model"]

    inputs = tokenizer(prompt_text, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0.0),
        temperature=temperature,
        top_p=top_p,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Remove echoed prompt if present
    if text.startswith(prompt_text):
        text = text[len(prompt_text):].strip()

    return text


In [ ]:

# If you prefer to use the Hugging Face Inference API, you can implement this:
def generate_with_hf_inference_api(prompt_text, api_token, model_name=MODEL_NAME, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P):
    """
    Example: use Hugging Face Inference API (requires HF token and model accessible via inference).
    This is provided as a reference; not used by default here.
    """
    import requests
    headers = {"Authorization": f"Bearer {api_token}"}
    payload = {
        "inputs": prompt_text,
        "parameters": {"max_new_tokens": max_new_tokens, "temperature": temperature, "top_p": top_p, "return_full_text": False},
    }
    url = f"https://api-inference.huggingface.co/models/{model_name}"
    resp = requests.post(url, headers=headers, json=payload, timeout=120)
    resp.raise_for_status()
    j = resp.json()
    # depends on response shape; attempt to extract generated_text
    if isinstance(j, list) and len(j) > 0 and "generated_text" in j[0]:
        return j[0]["generated_text"]
    # else return raw text
    return str(j)


In [ ]:
!pip install -U datasets --quiet



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 57.5 MB/s eta 0:00:00


In [ ]:
# stream_sample_natural_instructions.py


import json, hashlib, random, time
from itertools import islice
from datasets import load_dataset
from tqdm.auto import tqdm

# ---------- CONFIG ----------
DATASET_NAME = "Muennighoff/natural-instructions"
# OUTPUT_JSONL = "natural_instructions_1M_streamed.jsonl"
TARGET_COUNT = 1_000_000
SHUFFLE_BUFFER = 50_000
SEED = 42
WRITE_FLUSH_EVERY = 1000
# ----------------------------

def dedupe_key(row):
    # prefer id if present; else hash the input text
    if row.get("id"):
        return f"id::{row['id']}"
    inp = row.get("input") or row.get("inputs") or ""
    return "input_hash::" + hashlib.sha256((inp or "").encode("utf-8")).hexdigest()

def stream_and_write():
    print("Streaming dataset (no splits created/prepared). This avoids split creation errors.")
    ds_stream = load_dataset(DATASET_NAME, split="train", streaming=True)  # streaming=True avoids local split creation
    rng = random.Random(SEED)

    seen = set()
    buffer = []
    written = 0
    start = time.time()

    with open(OUTPUT_JSONL, "w", encoding="utf-8") as fout:
        for row in tqdm(ds_stream):
            # dedupe by id / input
            key = dedupe_key(row)
            if key in seen:
                continue
            seen.add(key)

            buffer.append(row)

            # When buffer is large enough, pop a random element and write it out:
            if len(buffer) >= SHUFFLE_BUFFER:
                i = rng.randrange(len(buffer))
                sample = buffer.pop(i)
                fout.write(json.dumps(sample, ensure_ascii=False) + "\n")
                written += 1
                if written % WRITE_FLUSH_EVERY == 0:
                    fout.flush()
                if written % 10000 == 0:
                    elapsed = time.time() - start
                    print(f"Written {written} (elapsed {elapsed:.1f}s). Buffer size {len(buffer)}.")
                if written >= TARGET_COUNT:
                    break

        # Flush remaining buffer in random order until we hit TARGET_COUNT
        rng.shuffle(buffer)
        for sample in buffer:
            if written >= TARGET_COUNT:
                break
            fout.write(json.dumps(sample, ensure_ascii=False) + "\n")
            written += 1
            if written % WRITE_FLUSH_EVERY == 0:
                fout.flush()

    print(f"Done. Wrote {written} examples to {OUTPUT_JSONL}")

# if __name__ == "__main__":
#     stream_and_write()


In [ ]:
# main_from_jsonl.py
import json
import time
from tqdm.auto import tqdm
import os

# ---------- CONFIG ----------
INPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgen.jsonl"   # your existing jsonl
OUTPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1000_promptgen.jsonl" # new file to store prompt-generation outputs
TARGET_COUNT = 1000    # how many examples to process
SAVE_EVERY = 100
# ----------------------------

def read_processed_ids(output_path):
    """Return a set of example_ids already written to the output file (for resume)."""
    if not os.path.exists(output_path):
        return set()
    processed = set()
    with open(output_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
                if "example_id" in rec:
                    processed.add(rec["example_id"])
            except Exception:
                continue
    return processed

def genprompts():
    # sanity checks
    if not os.path.exists(INPUT_JSONL):
        raise FileNotFoundError(f"Input JSONL not found: {INPUT_JSONL}")

    print(f"Resuming/starting prompt generation.\nInput: {INPUT_JSONL}\nOutput: {OUTPUT_JSONL}")
    processed_ids = read_processed_ids(OUTPUT_JSONL)
    print(f"Already processed examples found: {len(processed_ids)}")

    out_f = open(OUTPUT_JSONL, "a", encoding="utf-8")  # append mode for resumability

    processed = 0
    start_time = time.time()

    # Count total lines for nicer tqdm (optional; can be slow on huge files)
    try:
        with open(INPUT_JSONL, "r", encoding="utf-8") as fcount:
            total_lines = sum(1 for _ in fcount)
    except Exception:
        total_lines = None

    with open(INPUT_JSONL, "r", encoding="utf-8") as fin:
        for idx, raw in enumerate(tqdm(fin, total=total_lines)):
            if processed >= TARGET_COUNT:
                break

            try:
                row = json.loads(raw)
            except Exception:
                # skip malformed line but log
                print(f"Skipping malformed line idx={idx}")
                continue

            example_id = row.get("example_id") or row.get("id") or f"row_{idx}"
            if example_id in processed_ids:
                # skip already-processed example
                continue

            # gather fields
            definition = row.get("definition", "") or ""
            input_text = row.get("input", "") or row.get("inputs", "") or row.get("text", "") or ""

            # Build the prompt-generation instruction
            system_txt, user_txt = build_generation_system_and_user(definition, input_text)
            combined_prompt = system_txt + "\n\n" + user_txt

            # Call model to generate JSON (use your existing local/api function)
            try:
                model_out_text = generate_with_local_model(combined_prompt)
            except RuntimeError as e:
                # If local model fails, print and stop (or optionally switch to API)
                print("Local generation failed:", e)
                print("You can replace generate_with_local_model with an API call. Exiting loop.")
                break
            except Exception as e:
                # unexpected model error: save failure info and continue
                print(f"Model call error for example {example_id}: {e}")
                model_out_text = ""

            # Try to parse JSON from model output
            parsed = extract_json_from_text(model_out_text)
            if parsed is None:
                parsed = {"error_parse": True, "raw_text": model_out_text}

            # Build final record to save
            out_record = {
                "example_id": example_id,
                "source_idx": idx,
                "definition": definition,
                "input": input_text,
                "generation": {
                    "model": MODEL_NAME,
                    "params": {"max_new_tokens": MAX_NEW_TOKENS, "temperature": TEMPERATURE, "top_p": TOP_P},
                    "raw_text": model_out_text,
                    "parsed_json": parsed,
                },
                "processed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }

            # Write and flush periodically for safety
            out_f.write(json.dumps(out_record, ensure_ascii=False) + "\n")
            processed += 1
            processed_ids.add(example_id)

            if processed % SAVE_EVERY == 0:
                out_f.flush()
                elapsed = time.time() - start_time
                print(f"Processed {processed} new examples (elapsed {elapsed:.1f}s).")

    out_f.close()
    print(f"Done. New processed examples in this run: {processed}. Output file: {OUTPUT_JSONL}")

# if __name__ == "__main__":
#     genprompts()


In [ ]:




INPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgen.jsonl"
OUTPUT_JSONL = "/content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgeneratedNew2.jsonl"
TARGET_COUNT = 1_000_000
SAVE_EVERY = 100
# Batching configuration
BATCH_SIZE = 4
MAX_NEW_TOKENS = 32000         # per-example generation length
TEMPERATURE = 0.3
TOP_P = 0.9




In [ ]:


# tuning vars
PER_BATCH_MAX_SECONDS = 120
MAX_NEW_TOKENS = 4000


# Replacement flush_batch() — drop-in
import traceback, torch

In [ ]:
def genprompts():
    # sanity checks
    if not os.path.exists(INPUT_JSONL):
        raise FileNotFoundError(f"Input JSONL not found: {INPUT_JSONL}")

    print(f"Resuming/starting batched prompt generation.\nInput: {INPUT_JSONL}\nOutput: {OUTPUT_JSONL}")
    processed_ids = read_processed_ids(OUTPUT_JSONL)
    print(f"Already processed examples found: {len(processed_ids)}")

    out_f = open(OUTPUT_JSONL, "a", encoding="utf-8")  # append mode for resumability
    processed = 0
    start_time = time.time()
    batch_counter = 0

    # Ensure local model/tokenizer is loaded (lazy init)
    if "_LOCAL_GEN" not in globals() or not _LOCAL_GEN.get("ready", False):
        try:
            _ = generate_with_local_model("Warm up", max_new_tokens=1, temperature=0.0)
        except Exception:
            pass

    tokenizer = _LOCAL_GEN["tokenizer"]
    model = _LOCAL_GEN["model"]
    device = _LOCAL_GEN.get("device", "cuda" if torch.cuda.is_available() else "cpu")

    # Count file lines (optional)
    try:
        with open(INPUT_JSONL, "r", encoding="utf-8") as fcount:
            total_lines = sum(1 for _ in fcount)
    except Exception:
        total_lines = None

    batch_rows = []
    batch_meta = []

    def flush_batch():
        nonlocal processed, batch_rows, batch_meta, batch_counter
        if not batch_rows:
            return

        batch_counter += 1
        ids_sample = [m['example_id'] for m in batch_meta[:min(5, len(batch_meta))]]
        print(f"\n[batch {batch_counter}] START - size={len(batch_rows)}. Example IDs: {ids_sample} (showing up to 5)")

        t0 = time.time()
        # Build combined prompts for the batch
        combined_prompts = []
        for row, meta in zip(batch_rows, batch_meta):
            definition = meta["definition"]
            input_text = meta["input_text"]
            sys_txt, user_txt = build_generation_system_and_user(definition, input_text)
            combined_prompts.append(sys_txt + "\n\n" + user_txt)

        try:
            # Tokenize the batch (pad to longest)
            t_tok0 = time.time()
            tokenized = tokenizer(combined_prompts, return_tensors="pt", padding=True, truncation=True)
            t_tok1 = time.time()

            if device == "cuda":
                tokenized = {k: v.cuda() for k, v in tokenized.items()}

            input_len = tokenized["input_ids"].shape[1]
            ctx_limit = getattr(model.config, "max_position_embeddings", None)
            if ctx_limit is not None:
                dyn_max_new = max(1, min(MAX_NEW_TOKENS, ctx_limit - input_len))
            else:
                dyn_max_new = min(MAX_NEW_TOKENS, 1024)  # fallback safe cap

            print(f"[batch {batch_counter}] tokenized len={input_len}, dyn_max_new_tokens={dyn_max_new}, tokenization_time={(t_tok1-t_tok0):.3f}s")

            # Prepare generate kwargs:
            do_sample = (TEMPERATURE > 0.0)
            gen_kwargs = dict(
                **tokenized,
                max_new_tokens=dyn_max_new,
                do_sample=do_sample,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                max_time=PER_BATCH_MAX_SECONDS,   # stops generation after this many seconds (transformers)
            )
            if do_sample:
                gen_kwargs["temperature"] = float(TEMPERATURE)
                gen_kwargs["top_p"] = float(TOP_P)

            # Synchronize and run generation with explicit timing
            torch.cuda.empty_cache()
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            gen_start = time.time()
            with torch.no_grad():
                outputs = model.generate(**gen_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            gen_end = time.time()

            # Decode outputs (batch)
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

            # Write each example
            written = 0
            for meta, prompt_text, raw_out in zip(batch_meta, combined_prompts, decoded):
                example_id = meta["example_id"]
                idx = meta["source_idx"]
                gen_text = raw_out
                if gen_text.startswith(prompt_text):
                    gen_text = gen_text[len(prompt_text):].strip()
                parsed = extract_json_from_text(gen_text)
                if parsed is None:
                    parsed = {"error_parse": True, "raw_text": gen_text}

                out_record = {
                    "example_id": example_id,
                    "source_idx": idx,
                    "definition": meta["definition"],
                    "input": meta["input_text"],
                    "generation": {
                        "model": MODEL_NAME,
                        "params": {"batch_size": len(batch_rows), "max_new_tokens": dyn_max_new, "temperature": TEMPERATURE, "top_p": TOP_P},
                        "raw_text": gen_text,
                        "parsed_json": parsed,
                    },
                    "processed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                }
                out_f.write(json.dumps(out_record, ensure_ascii=False) + "\n")
                processed += 1
                processed_ids.add(example_id)
                written += 1

            out_f.flush()
            t1 = time.time()

            # GPU stats
            try:
                reserved_gb = torch.cuda.memory_reserved() / 1024**3
                allocated_gb = torch.cuda.memory_allocated() / 1024**3
            except Exception:
                reserved_gb = allocated_gb = 0.0

            print(f"[batch {batch_counter}] DONE - total={(t1-t0):.2f}s gen={(gen_end-gen_start):.2f}s token_time={(t_tok1-t_tok0):.3f}s written={written} processed_total={processed}")
            print(f"[batch {batch_counter}] GPU reserved={reserved_gb:.2f}GB allocated={allocated_gb:.2f}GB")

        except Exception as e:
            tb = traceback.format_exc()
            print(f"[batch {batch_counter}] ERROR during generation: {e}\n{tb}")
            # write an error record for each example so we don't stall forever
            for meta in batch_meta:
                example_id = meta["example_id"]
                rec = {
                    "example_id": example_id,
                    "source_idx": meta["source_idx"],
                    "definition": meta["definition"],
                    "input": meta["input_text"],
                    "generation": {
                        "model": MODEL_NAME,
                        "params": {"batch_size": len(batch_rows)},
                        "raw_text": "",
                        "error": str(e),
                    },
                    "processed_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                }
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                processed_ids.add(example_id)
            out_f.flush()
        finally:
            # always clear buffers so loop continues
            batch_rows = []
            batch_meta = []


    # Iterate input file and collect batches
    with open(INPUT_JSONL, "r", encoding="utf-8") as fin:
        for idx, raw in enumerate(tqdm(fin, total=total_lines)):
            if processed >= TARGET_COUNT:
                break

            try:
                row = json.loads(raw)
            except Exception:
                print(f"Skipping malformed line idx={idx}")
                continue

            example_id = row.get("example_id") or row.get("id") or f"row_{idx}"
            if example_id in processed_ids:
                continue

            definition = row.get("definition", "") or ""
            input_text = row.get("input", "") or row.get("inputs", "") or row.get("text", "") or ""

            batch_rows.append(row)
            batch_meta.append({"example_id": example_id, "source_idx": idx, "definition": definition, "input_text": input_text})

            if len(batch_rows) >= BATCH_SIZE:
                flush_batch()

            # periodic flush for long tail
            if processed % SAVE_EVERY == 0 and processed > 0:
                elapsed = time.time() - start_time
                print(f"Processed {processed} new examples (elapsed {elapsed:.1f}s). GPU reserved: {torch.cuda.memory_reserved()/1024**3:.2f}GB")

        # flush any remaining rows
        if batch_rows:
            flush_batch()

    out_f.close()
    print(f"Done. New processed examples in this run: {processed}. Output file: {OUTPUT_JSONL}")


In [ ]:
if __name__ == "__main__":
    genprompts()

Resuming/starting batched prompt generation.
Input: /content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgen.jsonl
Output: /content/drive/MyDrive/AdvNLP/natural_instructions_1M_promptgeneratedNew2.jsonl
Already processed examples found: 620


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[model] Loaded successfully.


  0%|          | 0/1000000 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



[batch 1] START - size=4. Example IDs: ['task002-b1ac77b00be3486a9ea87f1539494373', 'task024-a7ce167e239549b7ba853939e9f14b3d', 'task026-fe686e34b492494db22d4c3419110e4f', 'task028-6210cf73e9104858b6d861a110065c5a'] (showing up to 5)
[batch 1] tokenized len=1110, dyn_max_new_tokens=4000, tokenization_time=0.007s
[batch 1] DONE - total=23.67s gen=23.66s token_time=0.007s written=4 processed_total=4
[batch 1] GPU reserved=14.76GB allocated=13.51GB

[batch 2] START - size=4. Example IDs: ['task028-215f66dc05cd429ab6476966d63c5fa8', 'task028-814b685f602b41a08102a80372da715b', 'task024-642b72a5dbc14a268396e0bf9cc4740f', 'task002-17ddd092912e46daa2761182e0a05fc4'] (showing up to 5)
[batch 2] tokenized len=2085, dyn_max_new_tokens=4000, tokenization_time=0.010s
[batch 2] DONE - total=14.15s gen=14.13s token_time=0.010s written=4 processed_total=8
[batch 2] GPU reserved=15.90GB allocated=13.51GB

[batch 3] START - size=4. Example IDs: ['task044-949c5ae51deb4c8f9b37b1096c761c3e', 'task023-16f7